# Scraping et Interrogation des APIs (Codes-Barres)

L'objectif de ce notebook est de récupérer les valeurs nutritionnelles manquantes pour les produits de la base OpenFoodFacts en utilisant leur **code-barres (EAN-13)**.

Nous allons cibler :
1. **NumAlim** (API officielle française des données alimentaires)
2. **OpenFoodFacts Live API** (Au cas où le produit a été mis à jour par un contributeur depuis le dernier export)
3. **Scraping de sites e-commerce de Grande Distribution** (Auchan, Carrefour, Drive...)

In [ ]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
from tqdm import tqdm

# Pour forcer l'affichage de DataFrames dans Jupyter
from IPython.display import display

## 1. Chargement des codes-barres orphelins

In [ ]:
# On charge la base qu'on a exportée précédemment
file_path = "produits_sans_nutriscore.csv"

try:
    # dtype={'code': str} permet d'être sûr de ne perdre aucun zéro initial et d'éviter les floats
    df_orphelins = pd.read_csv(file_path, low_memory=False, dtype={'code': str})
    
    # Filtrer les produits qui n'ont pas de code (NaN ou vide)
    df_orphelins = df_orphelins.dropna(subset=['code'])
    df_orphelins = df_orphelins[df_orphelins['code'].str.strip() != '']
    
    codes_uniques = df_orphelins['code'].unique().tolist()
    print(f"✅ {len(codes_uniques):,} codes-barres distincts sont prêts à être interrogés.")
except FileNotFoundError:
    print(f"❌ Le fichier {file_path} est introuvable. Assurez-vous d'avoir exécuté 'analyse_nutri.ipynb' d'abord.")

## 2. API NumAlim

NumAlim est la plateforme numérique de référence du secteur alimentaire. Ce bloc contient l'infrastructure d'appel de base.
**Attention :** L'API NumAlim nécessite un Token (Clé d'API) que vous devez obtenir sur leur portail développeur.

In [ ]:
def fetch_api_numalim(ean, token="VOTRE_TOKEN_NUMALIM"):
    """
    Fonction pour appeler l'API de NumAlim.
    URL: https://api.numalim.fr/v1/produits/{ean} (URL fictive/hypothétique basée sur les standards REST)
    """
    # A adapter selon la VRAIE documentation de NumAlim (Endpoint exact)
    url = f"https://api.numalim.fr/v1/produits/{ean}"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }
    
    # Désactivé par sécurité tant qu'il n'y a pas de Token valide pour éviter de bloquer l'IP.
    if token == "VOTRE_TOKEN_NUMALIM":
        return None
        
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            data = response.json()
            # Extraire les nutriments du JSON retourné
            # return data.get('nutriments', {})
            return data
        elif response.status_code == 404:
            # Produit non connu chez NumAlim
            return None
        else:
            print(f"⚠️ NumAlim : Erreur HTTP {response.status_code} sur {ean}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"❌ NumAlim : Exception réseau sur {ean} - {e}")
        return None

## 3. Scraping Grande Surface (Drive / Supermarchés en Ligne)

Si l'API ferme ses portes, nous pouvons taper dans la barre de recherche d'une grande surface en injectant directement l'EAN. La plupart des e-commerces alimentaires associent le code EAN à la page produit HTML.

In [ ]:
def fetch_scraping_supermarche(ean):
    """
    Il s'agit d'une preuve de concept (PoC) avec BeautifulSoup4.
    Note: Beaucoup de drives utilisent du JavaScript lourd (React/Vue). 
    Si la requete ne ramène rien, il faudra remplacer `requests` par `Playwright` ou `Selenium`.
    """
    # Exemple générique de recherche (Adapter avec le domaine de Carrefour, Auchan, Leclerc, etc.)
    url = f"https://urldemon-supermarche.fr/recherche?q={ean}"
    
    # User-Agent très important pour ne pas être vu comme un robot illégitime instantanément
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
    }
    
    # RETURN anticipé car l'URL est fictive pour l'exemple
    return None
    
    try:
        response = requests.get(url, headers=headers, timeout=8)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # LOGIQUE DE PARSING A IMPLEMENTER ICI
            # 1. Chercher le tableau de nutrition (souvent classé via une class CSS 'nutrition-table' ou 'product-nutrients')
            # nutri_table = soup.find('table', class_='...nom...de...la...classe...')
            # 
            # 2. Extraire 'Energie', 'Sucres', 'Sel', 'Protéines'...
            # 
            # return dico_nutriments
            
            return "Produit_Scrapé_Avec_Succès" # Placeholder
        return None
    except Exception as e:
        print(f"❌ Scraping Exception sur {ean} : {e}")
        return None

## 4. API OpenFoodFacts (Live)

Il arrive qu'entre le moment où l'export CSV a été créé et le jour où l'on traite, les contributeurs aient mis à jour l'information. Taper l'API Live peut être une bonne solution de `Fallback`.

In [ ]:
def fetch_api_off_live(ean):
    url = f"https://world.openfoodfacts.org/api/v0/product/{ean}.json"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            # 'status' = 1 si le produit existe
            if data.get('status') == 1:
                produit = data.get('product', {})
                nutriments = produit.get('nutriments', {})
                
                # Si l'énergie y est, c'est bon signe.
                if 'energy-kcal_100g' in nutriments or 'energy_100g' in nutriments:
                    return {
                        'energy_100g': nutriments.get('energy_100g', None),
                        'energy-kcal_100g': nutriments.get('energy-kcal_100g', None),
                        'sugars_100g': nutriments.get('sugars_100g', None),
                        'salt_100g': nutriments.get('salt_100g', None),
                        'saturated-fat_100g': nutriments.get('saturated-fat_100g', None),
                        'proteins_100g': nutriments.get('proteins_100g', None),
                        'fiber_100g': nutriments.get('fiber_100g', None)
                    }
        return None
    except:
        return None

## 5. Exécution / Boucle Industrielle

Nous choisissons un tout petit échantillon (30 produits) car interroger des centaines de milliers de produits via API/Scraping de façon brute prendrait des jours et bloquerait votre connexion.

In [ ]:
if 'codes_uniques' in locals():
    echantillon = codes_uniques[:30] # On teste 30 produits
    resultats_enrichis = []
    
    print(f"⏳ Lancement de la pêche aux données manquantes sur {len(echantillon)} codes-barres...")
    
    for ean in tqdm(echantillon, desc="Recherche externe"):
        nutri_final = None
        source_trouvee = "Aucune"
        
        # Stratégie 1: NumAlim (Priorité Officielle)
        # nutri_numalim = fetch_api_numalim(ean)
        # si nutri_numalim:
        #     nutri_final = nutri_numalim
        #     source_trouvee = "NumAlim"
        
        # Stratégie 2: Scraping Drive (Si Numalim a échoué)
        if nutri_final is None:
            nutri_scrap = fetch_scraping_supermarche(ean)
            if nutri_scrap:
                nutri_final = nutri_scrap
                source_trouvee = "Scraping Supermarché"
                
        # Stratégie 3: OFF Live API en dernier recours
        if nutri_final is None:
            nutri_off = fetch_api_off_live(ean)
            if nutri_off is not None and len(nutri_off) > 0 and nutri_off.get('energy_100g') is not None:
                nutri_final = nutri_off
                source_trouvee = "OpenFoodFacts Live"
                
        # Sauvegarde du résultat
        resultats_enrichis.append({
            'code': ean,
            'source_recuperation': source_trouvee,
            'donnees': nutri_final
        })
        
        # PAUSE OBLIGATOIRE DE 1 SECONDE (Respect du Rate Limiting / Éviter d'être Banni)
        time.sleep(1)
        
    print("\n✅ Processus terminé. Aperçu des produits trouvés :")
    df_enrichi = pd.DataFrame(resultats_enrichis)
    display(df_enrichi.head(10))